# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | ** | 
| **Mehmet Kilinc 25135007** | *khaleelalaqel* |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [1]:
# Prompt 1: <CHAT-TITEL>
"""Imports en algemene instellingen voor het hele notebook."""
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import (GradientBoostingClassifier,
                              RandomForestClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             average_precision_score, confusion_matrix,
                             f1_score, make_scorer, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_predict, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Vaste instellingen zodat alle resultaten reproduceerbaar zijn
RANDOM_STATE = 42
N_FOLDS = 5
N_JOBS = -1  # gebruik alle CPU-kernen
DATA_DIR = Path("data")
SUBMISSION_DIR = Path("submissions")
SUBMISSION_DIR.mkdir(exist_ok=True)

# Waarschuwingen (bijv. over convergentie tijdens het tunen) maken de
# output onoverzichtelijk; daarom zetten we ze uit.
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)


In [ ]:
# Prompt 1: <CHAT-TITEL>
class StrokeDataset:
    """Laadt, controleert en schoont de stroke-data van Kaggle.

    Parameters
    ----------
    data_dir : Path
        Map waarin train.csv en test.csv staan.
    """

    TARGET = "stroke"
    ID_COLUMN = "id"
    NUMERIC_COLUMNS = ["age", "avg_glucose_level", "bmi"]
    # Van iedere binaire variabele houden we één dummy-kolom over
    REDUNDANT_COLUMNS = ["gender_Female", "ever_married_No",
                         "Residence_type_Rural"]
    DUMMY_GROUPS = ["gender", "ever_married", "work_type",
                    "Residence_type", "smoking_status"]

    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.train_raw = None
        self.test_raw = None
        self.train = None
        self.test = None
        self.test_ids = None
        self.cleaning_log = []

    def load(self):
        """Leest train.csv en test.csv in als DataFrames."""
        self.train_raw = pd.read_csv(self.data_dir / "train.csv")
        self.test_raw = pd.read_csv(self.data_dir / "test.csv")
        return self

    def column_overview(self):
        """Geeft per kolom datatype, uniek, ontbrekend, min en max."""
        data = self.train_raw
        return pd.DataFrame({
            "datatype": data.dtypes.astype(str),
            "aantal uniek": data.nunique(),
            "ontbrekend": data.isna().sum(),
            "minimum": data.min(),
            "maximum": data.max(),
        })

    def check_sklearn_requirements(self):
        """Controleert de minimale eisen van scikit-learn voor X en y."""
        train, test = self.train_raw, self.test_raw
        features = train.columns.drop(self.TARGET)
        checks = {
            "Alle kolommen zijn numeriek of boolean": all(
                pd.api.types.is_numeric_dtype(train[column])
                for column in train.columns),
            "Geen ontbrekende waarden in train":
                int(train.isna().sum().sum()) == 0,
            "Geen ontbrekende waarden in test":
                int(test.isna().sum().sum()) == 0,
            "Geen dubbele patiënten (rijen zonder id)":
                not train.drop(columns=self.ID_COLUMN).duplicated().any(),
            "Zelfde feature-kolommen in train en test":
                list(features) == list(test.columns),
            "Doelvariabele is binair (0/1)":
                set(train[self.TARGET].unique()) == {0, 1},
        }
        return pd.DataFrame({"voldaan": list(checks.values())},
                            index=list(checks.keys()))

    def readable_frame(self):
        """Zet de dummy-kolommen terug naar categorieën (voor de EDA)."""
        data = self.train_raw.copy()
        for group in self.DUMMY_GROUPS:
            prefix = group + "_"
            columns = [col for col in data.columns if col.startswith(prefix)]
            labels = data[columns].idxmax(axis=1).str.replace(
                prefix, "", regex=False)
            # Rijen waarin alle dummies False zijn: categorie 'Unknown'
            labels[~data[columns].any(axis=1)] = "Unknown"
            data[group] = labels
            data = data.drop(columns=columns)
        return data

    def clean(self):
        """Schoont train en test op (motivatie: paragraaf 3.7)."""
        train = self.train_raw.copy()
        test = self.test_raw.copy()

        # 1. Rijen met gender 'Other' zijn te zeldzaam om van te leren
        other_rows = train["gender_Other"]
        train = train.loc[~other_rows]
        self._log(f"{other_rows.sum()} rijen met gender 'Other' "
                  "verwijderd uit train")

        # 2. Id bewaren voor de submission en daarna als feature weghalen
        self.test_ids = test[self.ID_COLUMN].copy()
        drop_columns = ([self.ID_COLUMN, "gender_Other"]
                        + self.REDUNDANT_COLUMNS)
        train = train.drop(columns=drop_columns)
        test = test.drop(columns=drop_columns)
        self._log("kolommen verwijderd: " + ", ".join(drop_columns))

        # 3. Booleans omzetten naar 0/1 zodat alle features getallen zijn
        bool_columns = train.select_dtypes(include="bool").columns
        train[bool_columns] = train[bool_columns].astype(int)
        test[bool_columns] = test[bool_columns].astype(int)
        self._log(f"{len(bool_columns)} boolean-kolommen omgezet naar 0/1")

        self.train = train.reset_index(drop=True)
        self.test = test.reset_index(drop=True)
        return self

    def _log(self, message):
        """Bewaart een regel tekst over een opschoonstap."""
        self.cleaning_log.append(message)

    def get_features_and_target(self):
        """Geeft de features X en doelvariabele y van de opgeschoonde train."""
        features = self.train.drop(columns=self.TARGET)
        target = self.train[self.TARGET]
        return features, target

    def split(self, test_size=0.2):
        """Maakt een gestratificeerde train- en validatieset."""
        features, target = self.get_features_and_target()
        return train_test_split(features, target, test_size=test_size,
                                stratify=target,
                                random_state=RANDOM_STATE)

<br>

### Data inladen

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




vcguhctg